In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import precision_recall_curve, auc, accuracy_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns # plotting problem

!pip install contractions
import contractions # expanding contractions in text can't -> cannot

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
torch.backends.cudnn.deterministic = True
random.seed(21)
np.random.seed(21)
torch.manual_seed(21)
torch.cuda.manual_seed_all(21)

In [ ]:
import wandb
wandb.login(key = '02c8923278a3dc82932fafb9959cd6d7587dacc7')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
df = pd.read_json('/kaggle/input/amazon-product-review-spam-and-non-spam/Toys_and_Games/Toys_and_Games.json', lines=True).head(100000)
df = df.loc[:, ['reviewText', 'class']]
df

In [ ]:
df['reviewText'] = df['reviewText'].apply(lambda x: contractions.fix(x))
df.loc[:, 'reviewText'] = df['reviewText'].str.lower()

# \W represents Special characters like "$" and "!!!"
df.loc[:, 'reviewText'] = df['reviewText'].str.replace('\W', ' ')

# \d represents Numeric digits like "19.99"
df.loc[:, 'reviewText'] = df['reviewText'].str.replace('\d', ' ')

df

In [ ]:
def pr_auc_score(y_test, y_pred):
    precision, recall, _ = precision_recall_curve(y_test, y_pred)
    pr_auc = auc(recall, precision)
    return pr_auc

In [ ]:
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(df['reviewText'], y, test_size=0.33, random_state=21)

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = torch.tensor(texts, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        text = self.texts[index]
        text = text.unsqueeze(0)
        return text, self.labels[index]

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        # Initialize the hidden state and cell state with appropriate dimensions
        h0 = torch.zeros(1, batch_size, 128).to(device)
        c0 = torch.zeros(1, batch_size, 128).to(device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(GRUClassifier, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        # Initialize the hidden state with appropriate dimensions
        h0 = torch.zeros(1, batch_size, 128).to(device)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
def train_and_eval(train_loader, test_loader, model, criterion, optimizer):
    
    wandb.watch(model, criterion, log="all", log_freq=10)
    start_time = time.time()
    # Training loop
    num_epochs = 15
    for epoch in range(num_epochs):
        model.train()
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        wandb.log({"epoch": epoch,"loss": loss})
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    # Evaluation
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in test_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate accuracy and PR AUC score
    accuracy = accuracy_score(all_labels, all_preds)
    pr_auc = pr_auc_score(all_labels, all_preds)
    
    wandb.log({"test_accuracy": accuracy})
    wandb.log({"test_pr_auc": pr_auc})

    print(f'Accuracy: {accuracy:.4f}')
    print(f'AUC Score: {pr_auc:.4f}')
    
    elapsed_time = time.time() - start_time
    wandb.log({"run_time_sec": elapsed_time})
    wandb.finish()
    return model

# LSTM

### CountVectorizer 

In [ ]:
# Vectorization of text data (maybe change this to other embeddings or tokenization)
count_vectorizer = CountVectorizer(stop_words='english', max_df = 0.8, max_features = 2000)
X_train_vectorized = count_vectorizer.fit_transform(X_train).toarray()
X_test_vectorized = count_vectorizer.transform(X_test).toarray()

# Create PyTorch datasets
train_dataset = SpamDataset(X_train_vectorized, y_train)
test_dataset = SpamDataset(X_test_vectorized, y_test)

# Create PyTorch data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize device, model, optimizer, and loss function
model = LSTMClassifier(input_size=2000, hidden_size=128, num_layers=1, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

wandb.init(project='RNN Spam Detection', name='LSTM CountVectorizer')

train_and_eval(train_loader, test_loader, model, criterion, optimizer)

### TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_df=0.8, max_features=2000)
X_train_vectorized = vectorizer.fit_transform(X_train).toarray()
X_test_vectorized = vectorizer.transform(X_test).toarray()

# Create PyTorch datasets
train_dataset = SpamDataset(X_train_vectorized, y_train)
test_dataset = SpamDataset(X_test_vectorized, y_test)

# Create PyTorch data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize device, model, optimizer, and loss function
model = LSTMClassifier(input_size=2000, hidden_size=128, num_layers=1, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

wandb.init(project='RNN Spam Detection', name='LSTM TfidfVectorizer')

train_and_eval(train_loader, test_loader, model, criterion, optimizer)

# GRU

### CountVectorizer

In [ ]:
# Vectorization of text data (maybe change this to other embeddings or tokenization)
count_vectorizer = CountVectorizer(stop_words='english', max_df = 0.8, max_features = 2000)
X_train_vectorized = count_vectorizer.fit_transform(X_train).toarray()
X_test_vectorized = count_vectorizer.transform(X_test).toarray()

# Create PyTorch datasets
train_dataset = SpamDataset(X_train_vectorized, y_train)
test_dataset = SpamDataset(X_test_vectorized, y_test)

# Create PyTorch data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize device, model, optimizer, and loss function
model = GRUClassifier(input_size=2000, hidden_size=128, num_layers=1, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

wandb.init(project='RNN Spam Detection', name='GRU CountVectorizer')

train_and_eval(train_loader, test_loader, model, criterion, optimizer)

### Ifidf Vectorizer

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_df=0.8, max_features=2000)
X_train_vectorized = vectorizer.fit_transform(X_train).toarray()
X_test_vectorized = vectorizer.transform(X_test).toarray()

# Create PyTorch datasets
train_dataset = SpamDataset(X_train_vectorized, y_train)
test_dataset = SpamDataset(X_test_vectorized, y_test)

# Create PyTorch data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize device, model, optimizer, and loss function
model = GRUClassifier(input_size=2000, hidden_size=128, num_layers=1, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

wandb.init(project='RNN Spam Detection', name='GRU TfidfVectorizer')

train_and_eval(train_loader, test_loader, model, criterion, optimizer)